In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
images_path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")
print("Images downloaded to:", images_path)

Using Colab cache for faster access to the 'nih-chest-x-ray-14-224x224-resized' dataset.
Images downloaded to: /kaggle/input/nih-chest-x-ray-14-224x224-resized


In [7]:
import os

def looks_like_full_label_file(csv_path, min_rows=50000):

    try:
        df = pd.read_csv(csv_path, nrows=5)
    except Exception:
        return False
    if "Finding Labels" not in df.columns:
        return False
    # Cheap row count without loading the whole file
    with open(csv_path) as f:
        row_count = sum(1 for _ in f) - 1  # minus header
    return row_count >= min_rows

import pandas as pd

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(images_path) for f in files if f.endswith(".csv")]
print("CSV files found bundled with images:", csv_candidates)

labels_csv_path = None
for candidate in csv_candidates:
    if looks_like_full_label_file(candidate):
        labels_csv_path = candidate
        print(f"Using bundled labels file: {candidate}")
        break
    else:
        print(f"Rejected {candidate} — doesn't look like the full label set (wrong columns or too few rows)")

if labels_csv_path is None:
    print("No valid labels file bundled with the resized images — fetching Data_Entry_2017.csv "
          "from the original dataset instead (not the full 42GB of images, just this one file).")
    labels_dir = kagglehub.dataset_download("nih-chest-xrays/data", path="Data_Entry_2017.csv")
    labels_csv_path = labels_dir if labels_dir.endswith(".csv") else os.path.join(labels_dir, "Data_Entry_2017.csv")
    print("Labels downloaded to:", labels_csv_path)

img_dir_candidates = [root for root, dirs, files in os.walk(images_path) if any(f.lower().endswith((".png", ".jpg")) for f in files)]
IMAGES_DIR = img_dir_candidates[0]
print("Images folder:", IMAGES_DIR)

CSV files found bundled with images: ['/kaggle/input/nih-chest-x-ray-14-224x224-resized/BBox_List_2017_Official_NIH.csv', '/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv']
Rejected /kaggle/input/nih-chest-x-ray-14-224x224-resized/BBox_List_2017_Official_NIH.csv — doesn't look like the full label set (wrong columns or too few rows)
Using bundled labels file: /kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
Images folder: /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224


In [8]:
import pandas as pd
import numpy as np

full_labels_df = pd.read_csv(labels_csv_path)
print("Full dataset:", full_labels_df.shape)

N_SUBSET = 25000  # keeps a Colab session comfortably within free-tier limits
labels_df = full_labels_df.sample(n=N_SUBSET, random_state=42).reset_index(drop=True)
print("Subset:", labels_df.shape)

all_labels = labels_df["Finding Labels"].str.split("|").explode()
label_counts = all_labels.value_counts()
print(label_counts)

Full dataset: (112120, 12)
Subset: (25000, 12)
Finding Labels
No Finding            13551
Infiltration           4352
Effusion               2930
Atelectasis            2504
Nodule                 1431
Mass                   1259
Pneumothorax           1179
Consolidation           997
Pleural_Thickening      735
Cardiomegaly            621
Emphysema               537
Edema                   495
Fibrosis                373
Pneumonia               313
Hernia                   51
Name: count, dtype: int64


In [9]:
CONDITIONS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

missing = [c for c in CONDITIONS if c not in label_counts.index]
if missing:
    print("WARNING: these condition names don't match the dataset's labels:", missing)
    print("Available labels:", sorted(label_counts.index.tolist()))
else:
    print("All 14 condition names match.")

def labels_to_vector(finding_labels_str):
    labels = finding_labels_str.split("|")
    return np.array([1.0 if c in labels else 0.0 for c in CONDITIONS], dtype=np.float32)

label_matrix = np.stack(labels_df["Finding Labels"].apply(labels_to_vector).to_numpy())

print()
for i, c in enumerate(CONDITIONS):
    print(f"{c}: {int(label_matrix[:, i].sum())} positive examples in subset")

All 14 condition names match.

Atelectasis: 2504 positive examples in subset
Cardiomegaly: 621 positive examples in subset
Effusion: 2930 positive examples in subset
Infiltration: 4352 positive examples in subset
Mass: 1259 positive examples in subset
Nodule: 1431 positive examples in subset
Pneumonia: 313 positive examples in subset
Pneumothorax: 1179 positive examples in subset
Consolidation: 997 positive examples in subset
Edema: 495 positive examples in subset
Emphysema: 537 positive examples in subset
Fibrosis: 373 positive examples in subset
Pleural_Thickening: 735 positive examples in subset
Hernia: 51 positive examples in subset


In [10]:
from sklearn.model_selection import train_test_split

image_indices = labels_df["Image Index"].to_numpy()
train_idx, val_idx = train_test_split(np.arange(len(image_indices)), test_size=0.2, random_state=42)

train_files = image_indices[train_idx]
val_files = image_indices[val_idx]
train_labels = label_matrix[train_idx]
val_labels = label_matrix[val_idx]

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

Train: 20000, Val: 5000


In [11]:
import tensorflow as tf

IMG_SIZE = 224

def load_image(filename, label):
    path = tf.strings.join([IMAGES_DIR, filename], separator="/")
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])  # safety net if any file isn't exactly 224x224
    image = image / 255.0
    return image, label

def make_dataset(files, labels, shuffle=False, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if shuffle:
        ds = ds.shuffle(len(files), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, train_labels, shuffle=True)
val_ds = make_dataset(val_files, val_labels)

for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "batch label shape:", labs.shape)

batch image shape: (32, 224, 224, 3) batch label shape: (32, 14)


In [12]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

base_model = DenseNet121(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # phase 1: frozen base

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CONDITIONS), activation="sigmoid")(x)
model = models.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)
model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 14)             │        14,350 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,051,854 (26.90 MB)

 Trainable params: 14,350 (56.05 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Checkpointing every epoch is cheap insurance against a Colab disconnect —
# if the session drops mid-run, you don't lose the whole training run.
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase1.keras", save_best_only=True, monitor="val_loss"),
]

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    callbacks=callbacks,
)

Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 129s 161ms/step - auc: 0.5774 - loss: 0.1901 - val_auc: 0.6659 - val_loss: 0.1690
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 95ms/step - auc: 0.6533 - loss: 0.1758 - val_auc: 0.6970 - val_loss: 0.1654
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 67s 107ms/step - auc: 0.6780 - loss: 0.1724 - val_auc: 0.7061 - val_loss: 0.1655
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/step - auc: 0.6942 - loss: 0.1711 - val_auc: 0.7095 - val_loss: 0.1650
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 68s 109ms/step - auc: 0.6985 - loss: 0.1705 - val_auc: 0.7163 - val_loss: 0.1648
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 92ms/step - auc: 0.7093 - loss: 0.1698 - val_auc: 0.7112 - val_loss: 0.1658
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 95ms/step - auc: 0.7092 - loss: 0.1695 - val_auc: 0.7171 - val_loss: 0.1631
Epoch 8/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 60s 95ms/step - auc: 0.7164 - loss: 0.1692 - val_auc: 0.7182 - val_loss: 0.1628
Epoch 9/15
625/625 ━━━━━━━━━━━━━━━━━

In [14]:
base_model.trainable = True
for layer in base_model.layers[:-30]:  # keep most of DenseNet frozen; only fine-tune the top blocks
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower — avoid wrecking pretrained weights
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase2.keras", save_best_only=True, monitor="val_loss"),
]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    callbacks=callbacks,
)

Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 117s 136ms/step - auc: 0.6617 - loss: 0.1973 - val_auc: 0.6713 - val_loss: 0.1750
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 101ms/step - auc: 0.6911 - loss: 0.1823 - val_auc: 0.6921 - val_loss: 0.1702
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 99ms/step - auc: 0.7039 - loss: 0.1783 - val_auc: 0.7017 - val_loss: 0.1682
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 99ms/step - auc: 0.7150 - loss: 0.1771 - val_auc: 0.7044 - val_loss: 0.1673
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 99ms/step - auc: 0.7200 - loss: 0.1752 - val_auc: 0.7062 - val_loss: 0.1668
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 99ms/step - auc: 0.7243 - loss: 0.1751 - val_auc: 0.7082 - val_loss: 0.1663
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - auc: 0.7321 - loss: 0.1741 - val_auc: 0.7130 - val_loss: 0.1658
Epoch 8/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - auc: 0.7346 - loss: 0.1721 - val_auc: 0.7144 - val_loss: 0.1656
Epoch 9/15
625/625 ━━━━━━━━━━━━━━━━━━

In [15]:
from sklearn.metrics import roc_auc_score

y_true = val_labels
y_pred = model.predict(val_ds, verbose=0)

print(f"{'Condition':<20} {'AUC':>6} {'Positives in val':>18}")
aucs = []
for i, c in enumerate(CONDITIONS):
    n_pos = int(y_true[:, i].sum())
    if n_pos == 0 or n_pos == len(y_true):
        print(f"{c:<20} {'N/A':>6} {n_pos:>18}")
        continue
    auc = roc_auc_score(y_true[:, i], y_pred[:, i])
    aucs.append(auc)
    print(f"{c:<20} {auc:>6.3f} {n_pos:>18}")

print(f"\nMean AUC across conditions with enough data: {np.mean(aucs):.3f}")

Condition               AUC   Positives in val
Atelectasis           0.738                512
Cardiomegaly          0.761                131
Effusion              0.814                606
Infiltration          0.651                849
Mass                  0.687                230
Nodule                0.639                277
Pneumonia             0.650                 58
Pneumothorax          0.794                212
Consolidation         0.720                173
Edema                 0.819                100
Emphysema             0.803                 93
Fibrosis              0.721                 83
Pleural_Thickening    0.677                144
Hernia                0.796                 10

Mean AUC across conditions with enough data: 0.734


In [16]:
import json

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v2_densenet_transfer.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

Artifacts written:
total 34184
drwxr-xr-x 2 root root     4096 Sep 23 19:54 .
drwxr-xr-x 1 root root     4096 Sep 23 19:54 ..
-rw-r--r-- 1 root root      219 Sep 23 19:54 condition_names.json
-rw-r--r-- 1 root root 34989400 Sep 23 19:54 v2_densenet_transfer.keras


In [17]:
import shutil

shutil.make_archive("v2_imaging_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v2_imaging_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>